# Multimodal Fusion Model (R3D-18 + GAT-LSTM)

Trains a binary **drunk vs. sober** classifier by fusing:
- **Video branch:** 3D ResNet (R3D-18) on 100-frame clips
- **Landmark branch:** Graph Attention Network + LSTM/GRU on 68-point facial landmarks

**Configure:** Uncomment and set paths in the data-loading cells (`LANDMARKS_CSV`, `VIDEO_BASE_PATH`) before training.

In [ ]:
# Multimodal drunk/sober classifier: R3D-18 (video) + GAT-LSTM (landmarks)
# Configure dataset paths in the data-loading cells before training.

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
import cv2
import os
import re
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from torch_geometric.nn import GATConv
from torchvision.models.video import r3d_18


In [ ]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


In [ ]:
# Hyperparameters
IMG_SIZE = (224, 224)
SEQ_LENGTH = 100
BATCH_SIZE = 8
HIDDEN_DIM = 128
NUM_CLASSES = 2  


In [ ]:
# Safe evaluation function for landmarks
def safe_eval_nested(x):
    if isinstance(x, (list, np.ndarray)):
        return x
    if x is None or str(x).strip() in ["nan", "None", ""]:
        return np.nan
    try:
        evaluated = eval(x)
        if isinstance(evaluated, (list, np.ndarray)):
            if all(isinstance(frame, list) and len(frame) == 68 for frame in evaluated):
                return np.array(evaluated)
            else:
                return np.nan
        else:
            return np.nan
    except Exception:
        return np.nan


In [ ]:
# Process frames function
def process_frames(arr):
    if isinstance(arr, np.ndarray):
        arr = arr[:100]
        if len(arr) < 50:
            return None
        elif len(arr) < 100:
            arr = np.pad(arr, ((0, 100 - len(arr)), (0, 0), (0, 0)), mode='edge')
        return arr
    return None


In [ ]:
# --- Data paths (update before running) ---
# CSV produced by the landmark extraction notebook
# LANDMARKS_CSV = 'data/output/landmarks_ear_mar.csv'
# landmark_df = pd.read_csv(LANDMARKS_CSV)


In [ ]:
# Process video_name column (uncomment after loading CSV)
# landmark_df['video_name'] = landmark_df['video_name'].str.replace('/', '_')


In [ ]:
# landmark_df['video_name'] = landmark_df['video_name'].str.replace('/', '_').str.replace('’', "_").str.replace("'", "_").str.replace('"', "_").str.replace('”', "_").str.replace('“', "_")


In [ ]:
# Process landmarks column (uncomment after loading CSV)
# landmark_df['landmarks'] = landmark_df['landmarks'].apply(safe_eval_nested)
# landmark_df['landmarks'] = landmark_df['landmarks'].apply(process_frames)
# landmark_df = landmark_df.dropna(subset=['landmarks']).reset_index(drop=True)


In [ ]:
# landmarks_np = np.stack(landmark_df['landmarks'].values)


In [ ]:
# Combine data for subject-based splitting (uncomment after loading CSV)
# df_combined = pd.DataFrame({
#     'video_name': landmark_df['video_name'],
#     'landmarks': list(landmarks_np)
# })


In [ ]:
# df_combined.head()


In [ ]:

def get_facial_landmark_edges():
    """
    Define edges based on facial landmark connectivity.
    """
    edges = [
        # Jawline
        (0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8),
        (8, 9), (9, 10), (10, 11), (11, 12), (12, 13), (13, 14), (14, 15), (15, 16),
        
        # Eyebrows
        (17, 18), (18, 19), (19, 20), (20, 21),  # Right eyebrow
        (22, 23), (23, 24), (24, 25), (25, 26),  # Left eyebrow
        
        # Nose
        (27, 28), (28, 29), (29, 30),  # Nose ridge
        (31, 32), (32, 33), (33, 34), (34, 35),  # Nose base
        
        # Eyes
        (36, 37), (37, 38), (38, 39), (39, 40), (40, 41), (41, 36),  # Right eye
        (42, 43), (43, 44), (44, 45), (45, 46), (46, 47), (47, 42),  # Left eye
        
        # Mouth
        (48, 49), (49, 50), (50, 51), (51, 52), (52, 53), (53, 54), (54, 55), (55, 56),
        (56, 57), (57, 58), (58, 59), (59, 48),  # Outer lips
        (60, 61), (61, 62), (62, 63), (63, 64), (64, 65), (65, 66), (66, 67), (67, 60)  # Inner lips
    ]
    return edges


In [ ]:
def compute_optical_flow(frames):
    """Compute optical flow between consecutive frames."""
    flows = []
    prev_frame = cv2.cvtColor(frames[0], cv2.COLOR_RGB2GRAY)
    
    for frame in frames[1:]:
        curr_frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        flow = cv2.calcOpticalFlowFarneback(
            prev_frame, curr_frame, None, 
            pyr_scale=0.5, levels=3, winsize=15, 
            iterations=3, poly_n=5, poly_sigma=1.2, 
            flags=0
        )
        flows.append(flow)
        prev_frame = curr_frame
        
    return np.array(flows)


In [ ]:
import torch
import cv2
from torch.utils.data import Dataset
from torchvision import transforms

class MultiModalDataset(Dataset):
    def __init__(self, landmarks, video_paths, labels, seq_length=100, min_length=50, img_size=(224, 224), video_base_path=None):
        # video_base_path: root folder with drunk/ and sober/ subfolders (see README)
        # e.g. video_base_path = 'data/dataset'
        self.video_base_path = video_base_path

        # Pre-map video names to their full paths for efficient lookup
        self.video_path_map = {}
        for condition in ['drunk', 'sober']:
            condition_path = os.path.join(self.video_base_path, condition)
            for sub_folder in os.listdir(condition_path):
                sub_folder_path = os.path.join(condition_path, sub_folder)
                if not os.path.isdir(sub_folder_path):
                    continue
                for video_file in os.listdir(sub_folder_path):
                    self.video_path_map[f"{sub_folder}_{video_file}"] = os.path.join(sub_folder_path, video_file)

        # Store the data in the dataset
        self.landmarks = landmarks
        self.video_paths = video_paths
        self.labels = labels
        self.seq_length = seq_length
        self.min_length = min_length
        self.img_size = img_size
        self.transform = self.get_default_transform()
        self.edges = get_facial_landmark_edges()

        # Filter out video paths with fewer than min_length frames
        self.video_paths = [vp for vp in self.video_paths if self.is_video_valid(vp)]

    def get_default_transform(self):
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(self.img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        # Load landmarks
        landmarks = torch.tensor(self.landmarks[idx], dtype=torch.float32)
        edge_index = torch.tensor(self.edges, dtype=torch.long).t().contiguous()

        # Load video frames
        video_name = self.video_paths[idx]
        full_video_path = self.get_full_video_path(video_name)
        frames = self.load_video_frames(full_video_path)

        # Pad or trim frames to seq_length
        frames = self.process_frames(frames)

        # Apply transforms and stack
        frames_tensor = torch.stack([self.transform(frame) for frame in frames])

        # Reshape to (C, T, H, W) for 3D ResNet
        frames_tensor = frames_tensor.permute(1, 0, 2, 3)

        # Convert label to tensor
        label = self.labels[idx]

        return {
            'landmarks': landmarks,
            'frames': frames_tensor,
            'label': label,
            'edge_index': edge_index
        }

    def get_full_video_path(self, video_name):
        """Get the full video path using the pre-mapped dictionary."""
        if video_name in self.video_path_map:
            return self.video_path_map[video_name]
        print(f"Missing video: {video_name}")
        return None

    def load_video_frames(self, video_path):
        """Load frames from a video file."""
        cap = cv2.VideoCapture(video_path)
        frames = []

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # Convert frame to RGB
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        cap.release()
        return frames

    def process_frames(self, frames):
        """Pad or trim video frames to seq_length."""
        current_length = len(frames)

        # Skip video if it has fewer than min_length frames
        if current_length < self.min_length:
            print(f"Skipping video due to insufficient frames: {len(frames)} frames")
            return []  # Return an empty list which will skip this video

        # If we have enough frames, trim or pad the frames to seq_length
        if current_length >= self.seq_length:
            return frames[:self.seq_length]

        # Pad with the last frame if there are fewer frames
        padding = [frames[-1]] * (self.seq_length - current_length)
        return frames + padding

    def is_video_valid(self, video_name):
        """Check if a video has at least min_length frames."""
        full_video_path = self.get_full_video_path(video_name)
        frames = self.load_video_frames(full_video_path)
        return len(frames) >= self.min_length


In [ ]:
class SpatialTemporalModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, seq_len, graph_type):
        super(SpatialTemporalModel, self).__init__()

        self.num_layers = num_layers
        self.seq_len = seq_len
        self.graph_type = graph_type

        # Graph Attention Network (GATConv) for spatial modeling
        self.gnn = GATConv(input_dim, hidden_dim)

        # LSTM for temporal modeling
        self.lstm = nn.LSTM(
            hidden_dim, 
            hidden_dim, 
            batch_first=True
        )

    def forward(self, x, edge_index):
        """
        Args:
            x (Tensor): Shape [batch_size, seq_len, num_nodes, input_dim]
            edges (Tensor): Shape [2, num_edges] (edge indices)

        Returns:
            Tensor: LSTM hidden states of shape [batch_size, seq_len, hidden_dim]
        """
        batch_size, seq_len, num_nodes, input_dim = x.size()

        edge_index = edge_index.reshape(2, -1)  # Flatten edge indices to [2, num_edges]

        gnn_outputs = []

        # Process each temporal frame independently through the GNN
        for t in range(seq_len):
            frame = x[:, t, :, :]  # Extract t-th frame, shape: [batch_size, num_nodes, input_dim]
            frame = frame.reshape(-1, input_dim)  # Reshape for GATConv: [batch_size * num_nodes, input_dim]
            
            # Apply GATConv for spatial modeling
            gnn_out = self.gnn(frame, edge_index)  # Shape: [batch_size * num_nodes, hidden_dim]
            gnn_out = gnn_out.reshape(batch_size, num_nodes, -1).mean(dim=1)  # Aggregate node features: [batch_size, hidden_dim]
            gnn_outputs.append(gnn_out)

        # Stack GNN outputs across frames: [batch_size, seq_len, hidden_dim]
        temporal_input = torch.stack(gnn_outputs, dim=1)

        # Pass through the LSTM for temporal modeling
        lstm_out, _ = self.lstm(temporal_input)  # Shape: [batch_size, seq_len, hidden_dim]

        # Return the hidden states for use in the fusion model
        return lstm_out


In [ ]:
class VideoLandmarkFusionModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, num_layers, seq_len, graph_type):
        super(VideoLandmarkFusionModel, self).__init__()
        
        # ResNet3D for video feature extraction
        self.resnet3d = r3d_18(pretrained=True)
        self.resnet3d.avgpool = nn.AdaptiveAvgPool3d((1, 1, 1))  # Adaptive pooling for flexibility
        self.resnet3d.fc = nn.Identity()  # Remove fully connected layer

        # Spatial-Temporal Model for landmark features
        self.spatial_temporal = SpatialTemporalModel(input_dim, hidden_dim, num_layers, seq_len, graph_type)

        # GRU for temporal aggregation (lighter alternative to LSTM)
        self.temporal_rnn = nn.GRU(
            hidden_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=0.3
        )

        # Learnable weight for Weighted Fusion
        self.alpha_param = nn.Parameter(torch.tensor(0.5))  # Initial alpha for fusion

        # Fusion layers with batch normalization
        self.fusion_layers = nn.Sequential(
            nn.Linear(512 + hidden_dim, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(negative_slope=0.01),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(negative_slope=0.01),
            nn.Dropout(0.5)
        )

        # Fully connected layer for final classification
        self.fc_fusion = nn.Linear(128, num_classes)

    def forward(self, frames, edges, landmarks):
        # Extract video features
        video_features = self.resnet3d(frames).view(frames.size(0), -1)  # Shape: [batch_size, 512]

        # Extract landmark features using the Spatial-Temporal Model
        landmark_features = self.spatial_temporal(landmarks, edges)  # Shape: [batch_size, seq_len, hidden_dim]

        # Temporal aggregation using GRU
        landmark_features, _ = self.temporal_rnn(landmark_features)  # Shape: [batch_size, seq_len, hidden_dim]
        aggregated_landmark_features = landmark_features[:, -1, :]  # Last hidden state: [batch_size, hidden_dim]

        # Weighted fusion of video and landmark features
        alpha = torch.sigmoid(self.alpha_param)  # Normalize alpha to range [0, 1]
        fused_features = torch.cat([
            alpha * video_features, 
            (1 - alpha) * aggregated_landmark_features
        ], dim=-1)  # Shape: [batch_size, 512 + hidden_dim]

        # Pass through fusion layers
        fused_features = self.fusion_layers(fused_features)  # Shape: [batch_size, 128]

        # Final classification
        output = self.fc_fusion(fused_features)  # Shape: [batch_size, num_classes]

        return output


In [ ]:
def evaluate_model(model, val_loader, device):
    """
    Evaluates the model on the validation or test data loader.
    Args:
        model: PyTorch model to be evaluated.
        data_loader: DataLoader for validation or test data.
        device: Device (CPU or GPU) to run the evaluation.
    Returns:
        accuracy: The accuracy of the model on the provided data loader.
    """
    model.eval()  # Set the model to evaluation mode
    correct_predictions = 0
    total_samples = 0
    
    with torch.no_grad():  # Disable gradient calculations for evaluation
        for batch in val_loader:
            # Convert components to tensors and move to device
            landmarks = torch.tensor(batch['landmarks']).to(device)    # Node features for SpatialTemporalModel
            edges = torch.tensor(batch['edges']).to(device)           # Edge indices for the graph
            visual_frames = torch.tensor(batch['frames']).to(device)  # Video frames for ResNet3D

            labels = torch.tensor(batch['label']).to(device)          # Ground-truth labels
            
            # Forward pass
            outputs = model(visual_frames, edges, landmarks)
            _, predicted = torch.max(outputs, dim=1)
            
            # Update counters
            correct_predictions += (predicted == labels).sum().item()
            total_samples += labels.size(0)
    
    accuracy = (correct_predictions / total_samples) * 100
    return accuracy


In [ ]:
import matplotlib.pyplot as plt

def train_model_with_plots(model, train_loader, val_loader, criterion, optimizer, device, epochs=50):
    best_val_accuracy = 0.0

    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    for epoch in range(epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for batch in train_loader:
            # Move batch data to device
            frames = batch['frames'].to(device)
            landmarks = batch['landmarks'].to(device)
            labels = batch['label'].to(device)
            edge_index = batch['edge_index'].to(device)

            optimizer.zero_grad()
            outputs = model(frames, edge_index, landmarks)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct / total
        train_losses.append(train_loss)
        train_accuracies.append(train_accuracy)

        print(f"Epoch {epoch + 1}/{epochs}, Training Loss: {train_loss:.4f}, Training Accuracy: {train_accuracy:.2f}%")

        # Validation phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for batch in val_loader:
                frames = batch['frames'].to(device)
                landmarks = batch['landmarks'].to(device)
                labels = batch['label'].to(device)
                edge_index = batch['edge_index'].to(device)

                outputs = model(frames, edge_index, landmarks)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader)
        val_accuracy = 100 * correct / total
        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)

        print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%")

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy

    # Plotting the results
    plt.figure(figsize=(12, 5))

    # Loss plot
    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs + 1), train_losses, label="Train Loss")
    plt.plot(range(1, epochs + 1), val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss over Epochs")
    plt.legend()

    # Accuracy plot
    plt.subplot(1, 2, 2)
    plt.plot(range(1, epochs + 1), train_accuracies, label="Train Accuracy")
    plt.plot(range(1, epochs + 1), val_accuracies, label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title("Accuracy over Epochs")
    plt.legend()

    plt.tight_layout()
    plt.show()

    return best_val_accuracy


In [ ]:
def balance_datasets(paths_1, labels_1, groups_1, paths_2, labels_2, groups_2):
    min_size = min(len(paths_1), len(paths_2))

    # Select the first 'min_size' elements from both datasets
    balanced_paths = paths_1[:min_size] + paths_2[:min_size]
    balanced_labels = labels_1[:min_size] + labels_2[:min_size]
    balanced_groups = groups_1[:min_size] + groups_2[:min_size]

    return balanced_paths, balanced_labels, balanced_groups


In [ ]:
# valid_video_names = set(landmark_df['video_name'].values)  # uncomment after loading CSV


In [ ]:
import os

def collect_video_paths_with_groups(base_path, label, valid_video_names):
    video_names = []
    video_labels = []
    groups = []

    for root, _, files in os.walk(base_path):
        for file in files:
            if file.lower().endswith(('.mp4', '.avi', '.mkv')):
                folder_name = os.path.basename(root)
                clean_name = os.path.basename(file)
                full_video_name = f"{folder_name}_{clean_name}"
                if full_video_name in valid_video_names:
                    video_names.append(full_video_name)
                    video_labels.append(label)
                    groups.append(folder_name)

    return video_names, video_labels, groups

# Collect video paths — set your dataset paths before running
# valid_video_names = set(landmark_df['video_name'].values)
# drunk_paths, drunk_labels, drunk_groups = collect_video_paths_with_groups(
#     'data/dataset/drunk', 1, valid_video_names
# )
# sober_paths, sober_labels, sober_groups = collect_video_paths_with_groups(
#     'data/dataset/sober', 0, valid_video_names
# )


In [ ]:
# video_paths, labels, groups = balance_datasets(
#     drunk_paths, drunk_labels, drunk_groups,
#     sober_paths, sober_labels, sober_groups
# )


In [ ]:
# unique_paths, unique_labels, unique_groups, seen = [], [], [], set()
# for path, label, group in zip(video_paths, labels, groups):
#     if path not in seen:
#         unique_paths.append(path)
#         unique_labels.append(label)
#         unique_groups.append(group)
#         seen.add(path)
# video_paths, labels, groups = unique_paths, unique_labels, unique_groups


In [ ]:
# print(f"Total videos: {len(video_paths)}, Total groups: {len(set(groups))}")


In [ ]:
# filtered_landmark_df = landmark_df[landmark_df['video_name'].isin(video_paths)].reset_index(drop=True)


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader
import random


In [ ]:
# gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
# train_idx, val_idx = next(gss.split(video_paths, labels, groups))


In [ ]:
# train_paths = [video_paths[i] for i in train_idx]
# train_labels = [labels[i] for i in train_idx]
# val_paths = [video_paths[i] for i in val_idx]
# val_labels = [labels[i] for i in val_idx]


In [ ]:
# print(f"Train video DataFrame size: {len(train_paths)}")
# print(f"Validation video DataFrame size: {len(val_paths)}")


In [ ]:
# train_landmark_df = filtered_landmark_df[filtered_landmark_df['video_name'].isin(train_paths)].reset_index(drop=True)
# val_landmark_df = filtered_landmark_df[filtered_landmark_df['video_name'].isin(val_paths)].reset_index(drop=True)


In [ ]:
# train_np = np.stack(train_landmark_df['landmarks'].values)
# test_np = np.stack(val_landmark_df['landmarks'].values)


In [ ]:
# print(f"Train Landmark DataFrame size: {len(train_landmark_df)}")
# print(f"Validation Landmark DataFrame size: {len(val_landmark_df)}")


In [ ]:
# VIDEO_BASE_PATH = 'data/dataset'
# train_dataset = MultiModalDataset(train_np, train_paths, train_labels, video_base_path=VIDEO_BASE_PATH)
# val_dataset = MultiModalDataset(test_np, val_paths, val_labels, video_base_path=VIDEO_BASE_PATH)


In [ ]:
# print(f"Length of train_dataset: {len(train_dataset)}")


In [ ]:
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
# for batch in train_loader:
#     print(batch.keys())
#     break


In [ ]:
# input_dim = 2
# hidden_dim = 128
# num_classes = 2
# num_layers = 2
# seq_len = 100
# graph_type = 'facial'
# spatial_temporal_model = SpatialTemporalModel(input_dim, hidden_dim, num_layers, seq_len, graph_type).to(device)


In [ ]:
# resnet3d_model = r3d_18(pretrained=True).to(DEVICE)


In [ ]:
FUSION_DIM = 128 + 512  # Hidden dimension from SpatialTemporalModel + ResNet3D output


In [ ]:
# input_dim = 2
# hidden_dim = 128
# num_classes = 2
# num_layers = 2
# seq_len = 100
# graph_type = 'facial'
# model = VideoLandmarkFusionModel(
#     input_dim=input_dim,
#     hidden_dim=hidden_dim,
#     num_classes=num_classes,
#     num_layers=num_layers,
#     seq_len=seq_len,
#     graph_type=graph_type
# ).to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-5)
# best_accuracy = train_model_with_plots(
#     model, train_loader, val_loader, criterion, optimizer, device, epochs=25
# )
